In [ ]:
import os
import shutil
import subprocess
import sys

VIDEO_EXTS = (".mp4", ".mov", ".avi", ".mkv", ".flv", ".wmv", ".m4v", ".webm")


def find_ffmpeg():
    ffmpeg = shutil.which("ffmpeg")
    if ffmpeg:
        return ffmpeg

    candidates = [
        os.path.expandvars(r"%LOCALAPPDATA%\Microsoft\WinGet\Links\ffmpeg.exe"),
        os.path.expandvars(r"%ProgramFiles%\ffmpeg\bin\ffmpeg.exe"),
        os.path.expandvars(r"%USERPROFILE%\scoop\apps\ffmpeg\current\bin\ffmpeg.exe"),
    ]
    for path in candidates:
        if os.path.isfile(path):
            return path

    winget_packages = os.path.expandvars(r"%LOCALAPPDATA%\Microsoft\WinGet\Packages")
    if os.path.isdir(winget_packages):
        for root, _dirs, files in os.walk(winget_packages):
            if "ffmpeg.exe" in files and root.lower().endswith("bin"):
                return os.path.join(root, "ffmpeg.exe")

    return None


def upscale_to_4k(input_file, output_file, ffmpeg_path):
    if not output_file.lower().endswith(".mp4"):
        base, _ = os.path.splitext(output_file)
        output_file = base + ".mp4"

    command = [
        ffmpeg_path,
        "-y",
        "-i", input_file,
        "-vf", (
            "scale=3840:2160:force_original_aspect_ratio=decrease:flags=lanczos,"
            "pad=3840:2160:(ow-iw)/2:(oh-ih)/2,"
            "setsar=1,"
            "hqdn3d=1.5:1.5:6:6,"
            "eq=contrast=1.10:brightness=0.03:saturation=1.35:gamma=1.05,"
            "vibrance=intensity=0.25,"
            "unsharp=5:5:1.2:5:5:0.0"
        ),
        "-c:v", "libx264",
        "-preset", "slow",
        "-crf", "18",
        "-pix_fmt", "yuv420p",
        "-profile:v", "high",
        "-level", "5.1",
        "-c:a", "aac",
        "-b:a", "192k",
        "-movflags", "+faststart",
        output_file,
    ]

    print(f" Converting: {input_file}")
    print(f" Output:     {output_file}")
    print(" This can take a while for long videos...")

    try:
        subprocess.run(command, check=True, capture_output=True, text=True)

        ffprobe_path = (
            ffmpeg_path.replace("ffmpeg.exe", "ffprobe.exe")
            if ffmpeg_path.lower().endswith("ffmpeg.exe")
            else ffmpeg_path.replace("ffmpeg", "ffprobe")
        )
        probe = subprocess.run(
            [
                ffprobe_path,
                "-v", "error",
                "-select_streams", "v:0",
                "-show_entries", "stream=width,height",
                "-of", "csv=p=0:s=x",
                output_file,
            ],
            capture_output=True,
            text=True,
        )
        resolution = (probe.stdout or "").strip()
        if resolution == "3840x2160":
            print(f" Done (confirmed 4K): {output_file}")
        else:
            print(f" Done: {output_file} (reported resolution: {resolution or 'unknown'})")
        return True
    except subprocess.CalledProcessError as e:
        print(f" Failed to convert {input_file}")
        err = (e.stderr or e.stdout or str(e)).strip()
        if err:
            print(" FFmpeg error:")
            print(err[-2000:])
        return False
    except FileNotFoundError:
        print(" FFmpeg / FFprobe not found. Install FFmpeg and add it to PATH.")
        return False


def batch_upscale(folder_path, ffmpeg_path):
    converted = 0
    failed = 0
    skipped = 0

    for file_name in sorted(os.listdir(folder_path)):
        if not file_name.lower().endswith(VIDEO_EXTS):
            continue
        if "_4k" in os.path.splitext(file_name)[0].lower():
            continue

        input_path = os.path.join(folder_path, file_name)
        if not os.path.isfile(input_path):
            continue

        base, _ext = os.path.splitext(file_name)
        output_path = os.path.join(folder_path, f"{base}_4k.mp4")

        if os.path.exists(output_path):
            print(f" Skipping (already exists): {output_path}")
            skipped += 1
            continue

        if upscale_to_4k(input_path, output_path, ffmpeg_path):
            converted += 1
        else:
            failed += 1

    print(f"\n Finished. Converted: {converted}, Failed: {failed}, Skipped: {skipped}")


def main():
    ffmpeg_path = find_ffmpeg()
    if not ffmpeg_path:
        print(" FFmpeg is not installed (or not on PATH).")
        return 1

    print(f" Using FFmpeg: {ffmpeg_path}")

    path = input("Enter the path to a video file or a folder containing videos: ").strip().strip('"')
    if os.path.isfile(path):
        base, _ext = os.path.splitext(path)
        output_path = f"{base}_4k.mp4"
        if os.path.exists(output_path):
            # overwrite old 4K so new color/sharpness settings apply
            os.remove(output_path)
        return 0 if upscale_to_4k(path, output_path, ffmpeg_path) else 1
    if os.path.isdir(path):
        batch_upscale(path, ffmpeg_path)
        return 0

    print(" Invalid path. Please provide a valid file or folder path.")
    return 1


if __name__ == "__main__":
    code = main()
    in_notebook = "IPython" in sys.modules or "google.colab" in sys.modules
    if not in_notebook:
        raise SystemExit(code)

 Using FFmpeg: /usr/bin/ffmpeg
Enter the path to a video file or a folder containing videos: /content/drive/MyDrive/hero.mp4
 Converting: /content/drive/MyDrive/hero.mp4
 Output:     /content/drive/MyDrive/hero_4k.mp4
 This can take a while for long videos...
 Done (confirmed 4K): /content/drive/MyDrive/hero_4k.mp4
